# Scenario 2 — $100 annual DCA

Validation and exploration for the annual dollar-cost-averaging scenario: $100 invested on each yearly anniversary of the window start, ten deposits, terminal measured at month 120. Total contributed: $1,000.

Mirrors the structure of `01_lump_sum.ipynb` and uses the same `AnnualDCA100.compute_windows()` that `src/scenarios/annual_dca.py` exposes.

Run from the project root: `jupyter notebook notebooks/02_annual_dca.ipynb`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.scenarios.annual_dca import AnnualDCA100, HORIZON_YEARS
from src.analysis import rolling_window_returns

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.figsize'] = (11, 4)

## Load the monthly returns parquet

In [ ]:
df = (
    pd.read_parquet(ROOT / 'data' / 'processed' / 'monthly_returns.parquet')
    .set_index('date')
    .sort_index()
)
print(f'Rows: {len(df):,}')
print(f'Span: {df.index.min().date()} to {df.index.max().date()}')
df.head()

## Compute the rolling DCA windows

Each window starts at month `t` and runs for 10 years. Ten $100 deposits are made on yearly anniversaries (offsets 0, 12, …, 108 months) and each compounds from its deposit month to month 120. IRR is solved by bisection.


In [ ]:
scenario = AnnualDCA100()
windows = scenario.compute_windows(df, HORIZON_YEARS).dropna()
print(f'{HORIZON_YEARS}-year DCA windows: {len(windows):,}')
windows.head()

## Sanity check #1 — 1990s bull decade

For the window starting 1990-01-31, the lump-sum CAGR is ~17.99%. Annual-DCA IRR should be **lower** here because most of the capital arrives later in the bull run and earns less compounding.

In [ ]:
lump_nom = rolling_window_returns(df['nominal_return'], years=HORIZON_YEARS)
lump_real = rolling_window_returns(df['real_return'], years=HORIZON_YEARS)

for start in ['1990-01-31']:
    w = windows.loc[start]
    l = lump_nom.loc[start]
    print(f'Start {start} (end {w.end_date.date()})')
    print(f'  Lump-sum   $1,000 → ${l.terminal_value * 1000:,.0f}  ({l.cagr:.2%} CAGR)')
    print(f'  Annual DCA $1,000 → ${w.nominal_terminal:,.0f}  ({w.nominal_metric:.2%} IRR)')

## Sanity check #2 — Lost decade (2000-01-31)

This is the textbook DCA-vs-lump-sum case. The lump-sum CAGR was ~−0.38% (money invested in Jan 2000 was worth less than $1,000 in Jan 2010, on a nominal basis). Under annual DCA, terminal should still come in **above** $1,000 nominal — later deposits caught lower prices.

In [ ]:
for start in ['2000-01-31']:
    w = windows.loc[start]
    l = lump_nom.loc[start]
    print(f'Start {start} (end {w.end_date.date()})')
    print(f'  Lump-sum   $1,000 → ${l.terminal_value * 1000:,.0f}  ({l.cagr:.2%} CAGR)')
    print(f'  Annual DCA $1,000 → ${w.nominal_terminal:,.0f}  ({w.nominal_metric:.2%} IRR)')
    assert w.nominal_terminal > 1000, 'DCA terminal should beat break-even even when lump-sum CAGR is negative'

## Rolling IRR over time

In [ ]:
fig, ax = plt.subplots()
ax.plot(windows.index, windows['nominal_metric'], label='Nominal IRR', linewidth=1)
ax.plot(windows.index, windows['real_metric'], label='Real IRR', linewidth=1)
ax.axhline(0, color='black', linewidth=0.5)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax.set_title(f'{HORIZON_YEARS}-year rolling annual-DCA IRR by window start date')
ax.set_xlabel('Window start')
ax.set_ylabel('IRR')
ax.legend()
plt.tight_layout()
plt.show()

## Distribution of rolling IRRs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(windows['nominal_metric'], bins=40, edgecolor='white')
axes[0].set_title('Nominal IRR')
axes[0].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
axes[1].hist(windows['real_metric'], bins=40, edgecolor='white', color='C1')
axes[1].set_title('Real IRR')
axes[1].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

nom_neg = (windows['nominal_metric'] <= 0).mean()
real_neg = (windows['real_metric'] <= 0).mean()
print(f'Nominal IRR <= 0: {nom_neg:.2%} ({(windows["nominal_metric"] <= 0).sum():,} of {len(windows):,})')
print(f'Real IRR    <= 0: {real_neg:.2%} ({(windows["real_metric"]    <= 0).sum():,} of {len(windows):,})')

## Best and worst 10-year DCA windows

In [ ]:
print('Best nominal DCA windows:')
print(windows.nlargest(5, 'nominal_metric')[['end_date', 'nominal_metric', 'nominal_terminal']])
print('\nWorst nominal DCA windows:')
print(windows.nsmallest(5, 'nominal_metric')[['end_date', 'nominal_metric', 'nominal_terminal']])

In [ ]:
print('Best real DCA windows:')
print(windows.nlargest(5, 'real_metric')[['end_date', 'real_metric', 'real_terminal']])
print('\nWorst real DCA windows:')
print(windows.nsmallest(5, 'real_metric')[['end_date', 'real_metric', 'real_terminal']])

## DCA vs. lump-sum side by side

Scatter the annual-DCA IRR against the lump-sum CAGR for the same window starts. Bull decades sit below the 45° line (DCA loses because most capital arrives late); flat or down decades sit above it (DCA wins because later deposits buy at lower prices).

In [ ]:
joined = windows[['nominal_metric']].join(
    lump_nom['cagr'].rename('lump_cagr'),
    how='inner',
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(joined['lump_cagr'], joined['nominal_metric'], s=4, alpha=0.5)
lim = (joined.min().min() - 0.02, joined.max().max() + 0.02)
ax.plot(lim, lim, color='black', linewidth=0.5, linestyle='--', label='45° (DCA = lump)')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax.set_xlabel('Lump-sum CAGR (nominal)')
ax.set_ylabel('Annual-DCA IRR (nominal)')
ax.set_title('Annual-DCA IRR vs. lump-sum CAGR, by window start')
ax.legend()
plt.tight_layout()
plt.show()